# 06 — Generalization Evaluation of an SFT'd SLM Policy · Google Colab

**Phase 2 / Month 2** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

## What this notebook does
Loads an already-trained LoRA adapter from Drive (v1 or v2 from notebook 05),
then evaluates the SLM policy across a **grid of (season, building subset)
configurations** to answer thesis RQ2: *how well does the SLM agent
generalize to unseen buildings and seasons?*

For each configuration we collect:
* CityLearn district-level KPIs (same as nb05 §11)
* Action-token distribution (CHARGE / IDLE / DISCHARGE counts)
* SoC traces per building

All three are compared against No-Control and RBC baselines on the same
window, so seasonal effects on the baselines are subtracted out.

**No training in this notebook.** Inference only — much cheaper than nb05.
Each 2-week SLM rollout is ~110 min on L4; budget your config grid accordingly.

## § 0 — Config
> Edit this cell only. Everything else is fixed.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Repo ─────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/antonisbast/eclipse-thesis.git"
REPO_DIR    = "/content/eclipse-thesis"

# ── Model + adapter ──────────────────────────────────────────────────────
MODEL_ID:    str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:bool = True
MAX_SEQ_LEN: int  = 1024
MAX_NEW_TOKENS:int = 200

# Path to the LoRA adapter on Google Drive. Edit to point at v1 or v2.
# nb05 § 12 copies it to: /content/drive/MyDrive/eclipse-thesis/sft_adapters/lora_adapter
ADAPTER_PATH = "/content/drive/MyDrive/eclipse-thesis/sft_adapters/lora_adapter"
ADAPTER_TAG  = "v1"   # label that appears in result tables ("v1" or "v2")

# ── Eval grid ────────────────────────────────────────────────────────────
# The 2022_phase_all dataset starts at Month 7 (July) hour 0. Approximate
# season midpoints (2-week window centered, 336 hours each):
#   summer  ≈ t=1100  (mid-August)
#   autumn  ≈ t=3300  (mid-November)
#   winter  ≈ t=5500  (mid-February)
#   spring  ≈ t=7700  (mid-May)
EVAL_LEN = 336        # 2 weeks; drop to 168 for 1 week if pressed for time

# Edit this list to control the experiment grid. Each entry is one rollout.
# Comment out lines you don't want — every SLM rollout is ~110 min on L4.
CONFIGS = [
    # Seasonal sweep — full training building set
    {"name": "summer_full",  "start": 1100, "buildings": [0,1,2,3,4,5]},
    {"name": "autumn_full",  "start": 3300, "buildings": [0,1,2,3,4,5]},
    {"name": "winter_full",  "start": 5500, "buildings": [0,1,2,3,4,5]},
    {"name": "spring_full",  "start": 7700, "buildings": [0,1,2,3,4,5]},
    # Building-subset sweep — winter (hardest season) on partial sets
    {"name": "winter_half_a","start": 5500, "buildings": [0,1,2]},
    {"name": "winter_half_b","start": 5500, "buildings": [3,4,5]},
]

# ── Misc ─────────────────────────────────────────────────────────────────
HF_TOKEN: str = ""
SEED          = 42

import torch
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"✓ GPU: {g.name}  ({g.total_memory/1e9:.1f} GB VRAM)")
else:
    print("⚠ No GPU — set Runtime → Change runtime type → T4/L4")

n_slm = len(CONFIGS)
est_min = n_slm * EVAL_LEN / 168 * 55   # 55 min per 168-step rollout
print(f"Grid: {n_slm} SLM rollouts × {EVAL_LEN} steps  ≈  {est_min:.0f} min")


## § 1 — Installs
Same stack as nb05 v1. We don't need TRL's collator here (no training), so
we let pip pick whatever TRL version is compatible with Unsloth — simpler
and avoids the VRAM hit from disabling Unsloth's offload.

In [ ]:
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q CityLearn --no-deps
print("✓ CityLearn installed")

In [ ]:
!pip install -q --upgrade pip
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install -q --upgrade transformers peft trl accelerate bitsandbytes datasets sentencepiece torchao
!pip install -q triton==3.6.0
print("✓ Unsloth stack installed")

## § 2 — Clone repo + mount Drive

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

assert os.path.isdir(ADAPTER_PATH), (
    f"Adapter not found at {ADAPTER_PATH}. Either nb05 § 12 wasn't run, or\n"
    f"the path differs. List candidates:\n"
    f"  !ls /content/drive/MyDrive/eclipse-thesis/sft_adapters/\n"
)
print(f"✓ Adapter found at {ADAPTER_PATH}")

## § 3 — Load base Gemma + attach the saved LoRA adapter
We use Unsloth's `FastModel.from_pretrained` to load the base model in
4-bit, then `PeftModel.from_pretrained` to merge in the trained adapter
from Drive. `FastModel.for_inference` enables Unsloth's 2× generation
path.

In [ ]:
from unsloth import FastModel
from peft import PeftModel
import torch, time

torch.cuda.empty_cache()

_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base model loaded in {time.time()-_t0:.1f}s")

_t0 = time.time()
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
print(f"LoRA adapter attached in {time.time()-_t0:.1f}s")

FastModel.for_inference(model)
print("✓ Inference path enabled")

## § 4 — `SFTProvider` (identical to nb05 § 7)
Wraps the LoRA-adapted model behind the same `.step()` API used by every
other policy provider in this codebase.

In [ ]:
import logging, re
from src.sft import parse_actions, make_sft_prompt

_logger = logging.getLogger("sft_slm")
_ACTION_RE = re.compile(r"<action\s+building\s*=\s*\d+\s*>", re.IGNORECASE)
_is_gemma  = "gemma" in MODEL_ID.lower()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class SFTProvider:
    def __init__(self, model, tokenizer, max_new_tokens: int = 200):
        self.model         = model.eval()
        self.tokenizer     = tokenizer
        self.max_new_tokens= max_new_tokens
        self.label         = f"sft_{ADAPTER_TAG}:{MODEL_ID.split('/')[-1]}"
        self._device       = next(model.parameters()).device

    def complete(self, system: str, user: str, max_tokens=None) -> str:
        max_new = max_tokens or self.max_new_tokens
        if _is_gemma:
            msgs = [{"role": "user", "content": f"{system}\n\n{user}"}]
        else:
            msgs = [{"role": "system", "content": system},
                    {"role": "user",   "content": user}]
        enc = self.tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True,
        ).to(self._device)
        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        new_tokens = out[0][enc["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def step(self, state_text: str, system=None, n_buildings: int = 6,
             max_retries: int = 2, **kw):
        sys_p = system or make_sft_prompt(n_buildings)
        last  = ""
        for _ in range(max_retries):
            last = self.complete(sys_p, f"STATE:\n{state_text}")
            if _ACTION_RE.search(last):
                return parse_actions(last, n_buildings), last, False
        return [0.0]*n_buildings, last, True


slm = SFTProvider(model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
print(f"Provider ready: {slm.label}")

## § 5 — Env factory + per-config rollout helpers

In [ ]:
import citylearn
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicBatteryRBC
from src.env import (
    snapshot_state, MERLINReward, OBSERVATIONS_LLM, ACTIVE_ACTIONS, BUILDINGS,
)
from src.sft import render_state

warnings.filterwarnings("ignore")
np.random.seed(SEED); random.seed(SEED)

def make_env(start: int, length: int, buildings: list[int]) -> CityLearnEnv:
    env = CityLearnEnv(
        schema="citylearn_challenge_2022_phase_all",
        buildings=buildings,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=OBSERVATIONS_LLM,
        random_seed=SEED,
        simulation_start_time_step=start,
        simulation_end_time_step=start + length - 1,
    )
    env.reward_function = MERLINReward(env.get_metadata())
    return env


_TOK_RE = re.compile(r"<action building=\d+>([A-Z_0-9]+)</action>")

def run_slm_rollout(cfg: dict) -> dict:
    """Run a single SLM rollout. Returns env + KPI df + action histogram + soc traces."""
    import collections
    buildings = cfg["buildings"]
    n_b       = len(buildings)
    env       = make_env(cfg["start"], EVAL_LEN, buildings)
    env.reset()
    sys_p     = make_sft_prompt(n_b)
    tokens    = collections.Counter()
    n_fb      = 0
    t0        = time.time()
    done, t   = False, 0
    while not done:
        snap = snapshot_state(env)
        acts, raw, fb = slm.step(render_state(snap), system=sys_p, n_buildings=n_b)
        n_fb += int(fb)
        tokens.update(_TOK_RE.findall(raw))
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc)
        if t % 48 == 0:
            print(f"  [{cfg['name']}] t={t:4d}/{EVAL_LEN}  fb={n_fb}  {time.time()-t0:.0f}s")
        t += 1
    elapsed = time.time() - t0
    print(f"  [{cfg['name']}] done in {elapsed:.0f}s | fallbacks={n_fb}/{t}")
    return {
        "env":      env,
        "tokens":   tokens,
        "fb":       n_fb,
        "elapsed":  elapsed,
    }

def run_rbc_rollout(cfg: dict) -> CityLearnEnv:
    env = make_env(cfg["start"], EVAL_LEN, cfg["buildings"])
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env

def run_noop_rollout(cfg: dict) -> CityLearnEnv:
    env = make_env(cfg["start"], EVAL_LEN, cfg["buildings"])
    env.reset()
    n_b  = len(cfg["buildings"])
    done = False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env

print(f"CityLearn {citylearn.__version__}  |  {len(CONFIGS)} configs queued")

## § 6 — Run the grid
For each config: (1) no-op, (2) RBC, (3) SLM. Baselines finish in seconds;
SLM dominates the wall time. Intermediate KPIs are printed as we go so you
can catch obviously broken runs early.

In [ ]:
import pandas as pd

results = []   # one record per (config, policy) pair

for cfg in CONFIGS:
    print(f"\n{'='*60}\n {cfg['name']}  |  start={cfg['start']}  buildings={cfg['buildings']}  steps={EVAL_LEN}\n{'='*60}")

    # 1) baselines — fast
    print("  No-Control …");  env_no  = run_noop_rollout(cfg)
    print("  RBC …");         env_rbc = run_rbc_rollout(cfg)

    # 2) SLM — slow
    print("  SLM …")
    slm_out = run_slm_rollout(cfg)
    env_slm = slm_out["env"]

    # 3) collect
    for label, env in [("No-Control", env_no), ("RBC", env_rbc), (slm.label, env_slm)]:
        df = env.evaluate()
        df["policy"] = label
        df["config"] = cfg["name"]
        results.append(df)

    # 4) save action histogram (SLM only — baselines don't emit tokens)
    print(f"  action tokens (SLM): {dict(slm_out['tokens'])}")

    # 5) keep envs in dict for later SoC plotting
    cfg["_envs"]   = {"No-Control": env_no, "RBC": env_rbc, "SLM": env_slm}
    cfg["_tokens"] = slm_out["tokens"]
    cfg["_fb"]     = slm_out["fb"]
    cfg["_t_sec"]  = slm_out["elapsed"]

all_kpis = pd.concat(results)
print(f"\n✓ Grid complete. Total SLM wall time: {sum(c['_t_sec'] for c in CONFIGS)/60:.1f} min")

## § 7 — KPI comparison
Five district-level KPIs we care about, pivoted (config × policy). For each
config the SLM should ideally undercut both No-Control (=1.0 by definition
for cost/carbon/electricity) and RBC.

In [ ]:
import pandas as pd

KPI_KEEP = [
    "cost_total", "carbon_emissions_total", "electricity_consumption_total",
    "daily_peak_average", "ramping_average", "zero_net_energy",
]

dist = all_kpis[all_kpis["level"] == "district"]
wide = (dist[dist["cost_function"].isin(KPI_KEEP)]
        .pivot_table(index=["config", "cost_function"],
                     columns="policy", values="value")
        .round(3))

# SLM minus RBC — positive numbers mean SLM is WORSE on that KPI
slm_col = [c for c in wide.columns if c.startswith("sft_")][0]
wide["Δ SLM−RBC"] = (wide[slm_col] - wide["RBC"]).round(3)

print("District KPIs — lower is better, Δ<0 means SLM beats RBC:")
display(wide.loc[(slice(None), KPI_KEEP), :])

## § 8 — Action distribution per config
The critical diagnostic. If `CHARGE_*` columns are zero everywhere, the
policy is still mode-collapsed and any KPI wins are coincidental (the model
is effectively running a static "discharge whatever you have" rule).

In [ ]:
TOKEN_ORDER = ["CHARGE_100","CHARGE_80","CHARGE_60","CHARGE_40","CHARGE_20",
               "IDLE",
               "DISCHARGE_20","DISCHARGE_40","DISCHARGE_60","DISCHARGE_80","DISCHARGE_100"]

rows = []
for cfg in CONFIGS:
    tot = sum(cfg["_tokens"].values()) or 1
    row = {"config": cfg["name"]}
    for tok in TOKEN_ORDER:
        row[tok] = round(100 * cfg["_tokens"].get(tok, 0) / tot, 1)
    row["fallbacks"] = cfg["_fb"]
    rows.append(row)

action_df = pd.DataFrame(rows).set_index("config")
print("Action-token distribution per config (%):")
display(action_df)

# Coarse summary: how much of the policy is charging vs discharging?
summary = pd.DataFrame({
    "CHARGE %":    action_df[[c for c in action_df.columns if c.startswith("CHARGE")]].sum(axis=1),
    "IDLE %":      action_df["IDLE"],
    "DISCHARGE %": action_df[[c for c in action_df.columns if c.startswith("DISCHARGE")]].sum(axis=1),
}).round(1)
print("\nCoarse charge / idle / discharge split per config:")
display(summary)

## § 9 — Bar plot: action distribution per config

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4.5))
summary.plot(kind="bar", stacked=True, ax=ax,
             color=["#2ca02c", "#7f7f7f", "#d62728"])
ax.set_ylabel("% of action tokens")
ax.set_title(f"SLM action mix per config  ({slm.label})")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
ax.set_ylim(0, 100)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## § 10 — SoC trace per config (SLM only)
Small multiples: one row per config, one line per building. If SoC is
flat at ~0 for the whole window, the model is not actually using the
batteries (consistent with all-DISCHARGE token distributions).

In [ ]:
n_cfg = len(CONFIGS)
fig, axes = plt.subplots(nrows=n_cfg, ncols=1, figsize=(12, 2.4*n_cfg), sharex=False)
if n_cfg == 1: axes = [axes]

for ax, cfg in zip(axes, CONFIGS):
    env = cfg["_envs"]["SLM"]
    for i, b in enumerate(env.buildings):
        ax.plot(b.electrical_storage.soc, label=f"B{cfg['buildings'][i]}", linewidth=1.2)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("SoC")
    ax.set_title(f"{cfg['name']}  (start={cfg['start']}, n_b={len(cfg['buildings'])})")
    ax.legend(loc="upper right", ncol=len(cfg["buildings"]), fontsize=8)
    ax.grid(linestyle="--", alpha=0.4)
axes[-1].set_xlabel("Hour into eval window")
plt.tight_layout()
plt.show()

## § 11 — Persist results to Drive (optional)
KPI table + action mix + raw CSV — handy when comparing v1 vs v2 later.

In [ ]:
from pathlib import Path

stamp   = time.strftime("%Y%m%d_%H%M%S")
out_dir = Path(f"/content/drive/MyDrive/eclipse-thesis/eval_runs/{ADAPTER_TAG}_{stamp}")
out_dir.mkdir(parents=True, exist_ok=True)

all_kpis.to_csv(out_dir / "kpis_raw.csv", index=False)
wide.to_csv(out_dir / "kpis_pivot.csv")
action_df.to_csv(out_dir / "action_distribution.csv")
summary.to_csv(out_dir / "action_summary.csv")

# Compact manifest of what we ran
manifest = {
    "adapter_path":  ADAPTER_PATH,
    "adapter_tag":   ADAPTER_TAG,
    "model_id":      MODEL_ID,
    "eval_len":      EVAL_LEN,
    "configs":       [{k: v for k, v in c.items() if not k.startswith('_')}
                      for c in CONFIGS],
    "slm_wall_time_sec": {c["name"]: c["_t_sec"] for c in CONFIGS},
    "fallbacks":     {c["name"]: c["_fb"] for c in CONFIGS},
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"✓ Saved to {out_dir}")